# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Begin on your hands and knees. Alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From a hands-and-knees position, extend opposite arm and leg simultaneously while engaging your core. Hold for about 5 seconds, then switch sides. Complete 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds. Repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent and arms crossed over your chest. Engage your stomach muscles to lift your shoulders off the floor briefly, then lower down. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** While lying on your back, pull one knee toward your chest, keeping the other foot flat on the floor. Hold for 15-30 

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Proper sleep is crucial for physical recovery, as the body repairs tissues and regenerates during deep sleep stages. It also plays a vital role in mental well-being and cognitive function, facilitating memory consolidation and learning. Adequate sleep helps strengthen the immune system, making you more resistant to illnesses, and supports hormonal regulation that influences growth and appetite. Conversely, poor sleep or sleep disorders like insomnia can negatively affect overall health, leading to issues such as weakened immunity, mental health problems, and problems with concentration and mood. Therefore, maintaining good sleep habits and ensuring sufficient quality sleep are essential for optimal health.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Giving a gentle massage to the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing, progressive muscle relaxation, or grounding techniques\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese methods can help alleviate headaches and reduce stress naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Aim for 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep positively impacts overall health in several important ways. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as keeping the room cool, dark, and quiet—help improve sleep quality. Good sleep hygiene practices, like establishing relaxing bedtime routines and limiting screen time before bed, promote restorative sleep. Adequate and restful sleep is essential for supporting the immune system, mental health, and proper nutrient absorption, all of which contribute to overall wellness. Conversely, poor sleep or insomnia can negatively affect your health, highlighting the importance of good sleep habits for maintaining overall health.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. For headaches, managing triggers like dehydration, stress, or poor sleep can help, and remedies such as herbal teas (like chamomile or valerian root), staying hydrated, and ensuring adequate rest may also be beneficial.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

BM25 outperforms embedding-based retrieval in situations where queries contain precise, exact terminology that appears verbatim in the source documents. Unlike vector search, which retrieves based on semantic similarity and can drift toward related but imprecise concepts, BM25 anchors on literal string matching which makes it more reliable when the exact phrasing matters.

For example, in a health and wellness knowledge base, a query like "What is progressive muscle relaxation?" demonstrates this advantage clearly. If the document contains that exact phrase, BM25 retrieves it instantly and precisely. Vector search would likely perform well here too, but BM25's keyword matching is faster and leaves no ambiguity.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back upward (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n2. **Bird Dog:** From hands and knees, extend opposite arm and leg while engaging your core. Hold each extension for 5 seconds, then switch sides. Do 10 repetitions per side.\n3. **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs and tilt your pelvis upward to flatten your lower back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours for adults, supports these restorative processes and helps maintain optimal health. Conversely, poor sleep or conditions like insomnia can negatively impact health, highlighting the importance of creating a good sleep environment and managing sleep difficulties.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet environment, gentle massage of the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule. Additionally, practicing deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music can help relieve stress and reduce headache symptoms.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n4. Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\n5. Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and prevent future episodes of lower back pain. Ho

In [26]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. During sleep, the body performs essential functions such as repairing tissues, consolidating memories, and releasing hormones that regulate growth and appetite. Adequate and quality sleep supports physical health, mental well-being, cognitive function, immune system strength, and recovery from illnesses. Poor sleep or sleep disorders like insomnia can lead to health problems, increased stress, and a decline in overall wellness. Therefore, maintaining good sleep hygiene and creating a sleep-friendly environment are crucial for sustaining good health.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Staying well-hydrated by drinking plenty of water\n- Applying warm or cold compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Engaging in deep breathing exercises\n- Practicing mindfulness, meditation, or relaxation techniques\n- Maintaining a regular sleep schedule and establishing a calming evening routine\n- Taking short walks in nature\n- Listening to calming music\n\nThese methods can help alleviate stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

A single query can miss relevant documents due to vocabulary mismatch. For example, if a user's query says "back pain" but the relevant chunk in the document says "lumbar discomfort", a single embedding vector for back pain will likely not pull that document chunk to the top for the LLM to generate an answer. A reformulated query that references "lumbar discomfort" may actually retrieve the proper information from the document to satisfy the original back pain query. Additionally, multiple reformulations search the embedding space from different angles and thus reach different regions where relevant documents cluster.

Through multiple reformulations of a user query, the union and deduplication of results directly increases recall as documents that scored low for the original query but high for a reformulation are surfaced from the corpus. Ultimately, the LLM is able to handle all the retrieved content and filter out what's relevant during generation, which is acceptable even if precision decreases as a result.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the information provided, exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: on hands and knees, alternate arching and sagging your back (10-15 repetitions)\n- Bird Dog: on hands and knees, extend opposite arm and leg, hold briefly, then switch sides (10 repetitions per side)\n- Partial Crunches: lying on your back with knees bent, lift shoulders off the floor while tightening your stomach muscles (8-12 repetitions)\n- Knee-to-Chest Stretch: lie on your back and pull one knee toward your chest, hold, then switch legs (15-30 seconds per leg)\n- Pelvic Tilts: lying on your back with knees bent, flatten your back against the floor by engaging your abs and tilting your pelvis (hold for 10 seconds, repeat 8-12 times)\n\nThese gentle stretching and strengthening exercises can help alleviate lower back pain and may prevent future issues.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of quality sleep per night. Proper sleep hygiene, such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment, can improve sleep quality. Adequate sleep supports body repair, memory, learning, and hormone regulation, all of which are crucial for overall health.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in relaxation techniques like progressive muscle relaxation, mindfulness or meditation, and light physical activity such as gentle stretching or yoga. Additionally, techniques like taking a short walk, listening to calming music, and practicing grounding techniques can help manage stress. \n\nFor headaches specifically, natural remedies include staying well-hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, giving gentle massage to the temples and neck, and using essential oils such as peppermint or lavender. Maintaining a regular sleep schedule and managing common triggers like dehydration and stress can also help prevent headaches.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle stretching and strengthening move

In [40]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health and well-being. According to the information provided, sleep is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and regulates hormones that influence growth and appetite. Adequate sleep (typically 7-9 hours per night) helps support immune function, reduces the risk of chronic health issues, and promotes emotional stability. Poor sleep or sleep disturbances, such as insomnia, can negatively affect health by impairing immune response, increasing stress, and contributing to mental health problems. Therefore, maintaining good sleep habits and creating a restful sleep environment are essential for sustaining overall health.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated to prevent dehydration.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Gently massaging the temples and neck.\n- Using essential oils such as peppermint or lavender.\n- Maintaining a regular sleep schedule.\n- Practicing immediate stress relief techniques like deep breathing, progressive muscle relaxation, grounding exercises, taking short walks, or listening to calming music.\n\nAdditionally, managing stress through mindfulness, meditation, and maintaining healthy lifestyle habits can help reduce the frequency and severity of headaches caused by stress.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [44]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include gentle stretching and strengthening movements. According to the provided information, some recommended exercises are:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over your chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nThese exercise

In [48]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a profound impact on overall health. According to the provided information, during sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night for adults—supports physical health, mental well-being, and cognitive functions. Proper sleep hygiene, such as maintaining a consistent sleep schedule, creating a comfortable environment, and limiting screen exposure before bed, helps improve sleep quality. Conversely, poor sleep or sleep disturbances like insomnia can negatively affect hormone regulation, immune function, mood, and cognitive performance, highlighting the importance of good sleep habits for overall wellness.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress and headaches include:\n\n- For stress:\n  - Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n  - Progressive muscle relaxation\n  - Grounding techniques (e.g., identifying things you see, hear, feel, smell, and taste)\n  - Taking short walks, preferably in nature\n  - Listening to calming music\n  - Practicing mindfulness and meditation (such as focused attention, body scan, loving-kindness, walking meditation)\n  - Engaging in hobbies and leisure activities\n  - Maintaining social connections and support networks\n\n- For headaches:\n  - Drinking plenty of water to stay hydrated\n  - Applying cold or warm compresses to the head or neck\n  - Resting in a dark, quiet room\n  - Gentle massage of temples and neck\n  - Using essential oils like peppermint or lavender\n  - Consuming caffeine in small amounts (with caution, as it can help or hurt)\n  - Keeping a regular sleep schedule\n\nImplementing these natural approaches c

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

Semantic chunking would look at short and highly repetitive sentences such as those in FAQs and group them together because their embedding vectors are very close to each other since most of the questions center around the same topic with slightly different phrasing. They will likely get lumped into one giant chunk and this might cause problems because a user can ask a question that's similar to all of the FAQ entries or just one of them.

To address this, you can adjust the algorithm to use gradient thresholding where similarity changes abruptly are detected instead of comparing each sentence against a global statistical baseline. FAQ topic boundaries work exactly like this with slight varied similarity changes. Additionally, the breakpoint threshold on the gradient should be lowered so even small changes are enough to trigger a split, which allows each FAQ question to be within its own focused chunk.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [50]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from datasets import Dataset
import pandas as pd

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Golden Dataset Generation
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
testset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)
golden_dataset = testset.to_pandas()
golden_dataset.head()

# Evaluation Helper Function
def evaluate_retriever(retriever, golden_dataset, retriever_name):
    results = []
    
    for _, row in golden_dataset.iterrows():
        question = row["user_input"]
        ground_truth = row["reference"]
        
        retrieved_docs = retriever.invoke(question)
        contexts = [doc.page_content for doc in retrieved_docs]
        
        results.append({
            "user_input": question,
            "retrieved_contexts": contexts,
            "reference": ground_truth
        })
    
    dataset = Dataset.from_list(results)
    score = evaluate(dataset, metrics=[context_precision, context_recall])
    
    print(f"\n{retriever_name} Results:")
    return score.to_pandas()

# Run each retriever
naive_results = evaluate_retriever(naive_retriever, golden_dataset, "Naive RAG")
bm25_results = evaluate_retriever(bm25_retriever, golden_dataset, "BM25")
compression_results = evaluate_retriever(compression_retriever, golden_dataset, "Contextual Compression")
multiquery_results = evaluate_retriever(multi_query_retriever, golden_dataset, "Multi-Query")
parent_results = evaluate_retriever(parent_document_retriever, golden_dataset, "Parent Document")
ensemble_results = evaluate_retriever(ensemble_retriever, golden_dataset, "Ensemble")

# Compile Final Comparison
all_results = {
    "Naive RAG": naive_results,
    "BM25": bm25_results,
    "Contextual Compression": compression_results,
    "Multi-Query": multiquery_results,
    "Parent Document": parent_results,
    "Ensemble": ensemble_results
}

comparison = pd.DataFrame([
    {
        "Retriever": name,
        "Context Precision": results["context_precision"].mean(),
        "Context Recall": results["context_recall"].mean()
    }
    for name, results in all_results.items()
])

comparison

/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_11802/3377730038.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_11802/3377730038.py:6: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall
/var/folders/c3/gpyjtk951l19x3vj98kvm25r0000gn/T/ipykernel_11802/3377730038.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; ll

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


Parent Document Results:


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


Ensemble Results:


,Retriever,Context Precision,Context Recall
0,Naive RAG,0.584679,0.741667
1,BM25,0.408333,0.350000
2,Contextual Compression,0.800000,0.575000
3,Multi-Query,0.531823,0.875000
4,Parent Document,0.750000,0.716667
5,Ensemble,0.466038,0.841667


Across the six retrieval methods evaluated on the health and wellness knowledge base, Contextual Compression achieved the highest context precision at 0.80, meaning the chunks it returned were the most relevant relative to what was retrieved. However, Multi-Query led in context recall at 0.875, meaning it was the best at surfacing all the relevant information that existed in the corpus. This aligns with the theoretical strengths of each method: Contextual Compression filters aggressively for quality while Multi-Query casts a wider net through query reformulation. BM25 performed the weakest across both metrics, which makes sense for this dataset since health and wellness content uses varied terminology and paraphrasing rather than the exact repeated terminology that BM25 thrives on. Ensemble had strong recall at 0.84 but lower precision at 0.47, suggesting it surfaces a lot of relevant content but also brings in noise. For this particular dataset, Contextual Compression is the best overall retriever if precision and answer quality are the priority, while Multi-Query is the better choice if comprehensive recall matters more. From a cost and latency perspective, BM25 and Multi-Query were the fastest to evaluate while Contextual Compression and Parent Document were the slowest, reflecting the additional reranking and document reconstruction steps each method requires.
